# 🧠 PyTorch Computer Vision: Digit Recognizer (MNIST)

Welcome to your first Deep Learning & Computer Vision project on Kaggle! In this notebook, we construct a **Convolutional Neural Network (CNN)** using **PyTorch** to classify 28x28 grayscale images of handwritten digits (0 through 9).

### 🎯 Objectives
1. **Image Inspection**: Reshape 784 flat pixel columns into $(1, 28, 28)$ image tensors and plot digit samples.
2. **CNN Architecture**: Build a 2-layer Convolutional Neural Network with Batch Normalization, Max Pooling, Dropout, and Linear classifier in PyTorch.
3. **Training & Validation**: Train over 5 epochs using `Adam` optimizer & `CrossEntropyLoss`, tracking accuracy.
4. **Visual Diagnostics**: Plot training loss curves and a 10x10 confusion matrix across digit classes.
5. **Kaggle Submission Export**: Generate predictions on `test.csv` and export a formatted `submission.csv` (`ImageId`, `Label`).

## 1. Setup & Environment Configuration

In [ ]:
import os
import glob
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ PyTorch version {torch.__version__} initialized on device: {device}")

## 2. Data Acquisition & Inspection

In [ ]:
def generate_synthetic_digits(n_train=2000, n_test=1000):
    """Generates synthetic 28x28 digit images for demonstration fallback."""
    np.random.seed(42)
    # Create simple synthetic pixel arrays
    y_train = np.random.randint(0, 10, size=n_train)
    X_train = np.random.randint(0, 256, size=(n_train, 784))
    X_test = np.random.randint(0, 256, size=(n_test, 784))
    
    train_df = pd.DataFrame(X_train, columns=[f'pixel{i}' for i in range(784)])
    train_df.insert(0, 'label', y_train)
    
    test_df = pd.DataFrame(X_test, columns=[f'pixel{i}' for i in range(784)])
    test_df.insert(0, 'ImageId', np.arange(1, n_test + 1))
    return train_df, test_df

# Search for Kaggle or local dataset
train_matches = glob.glob('/kaggle/input/**/train.csv', recursive=True) + glob.glob('../input/**/train.csv', recursive=True) + glob.glob('./**/train.csv', recursive=True)
test_matches = glob.glob('/kaggle/input/**/test.csv', recursive=True) + glob.glob('../input/**/test.csv', recursive=True) + glob.glob('./**/test.csv', recursive=True)

# Filter specifically for digit recognizer if multiple train.csv exist
digit_train = [f for f in train_matches if 'digit' in f.lower() or 'mnist' in f.lower()]
digit_test = [f for f in test_matches if 'digit' in f.lower() or 'mnist' in f.lower()]

if len(digit_train) > 0 and len(digit_test) > 0:
    train_df = pd.read_csv(digit_train[0])
    test_df = pd.read_csv(digit_test[0])
    print(f"✅ Loaded Digit Recognizer dataset: {digit_train[0]}")
elif len(train_matches) > 0 and len(test_matches) > 0 and 'label' in pd.read_csv(train_matches[0], nrows=2).columns:
    train_df = pd.read_csv(train_matches[0])
    test_df = pd.read_csv(test_matches[0])
    print(f"✅ Loaded dataset: {train_matches[0]}")
else:
    print("ℹ️ Dataset not attached. Generating synthetic 28x28 digit data...")
    train_df, test_df = generate_synthetic_digits(2000, 1000)

print(f"Train DataFrame Shape: {train_df.shape}")
print(f"Test DataFrame Shape:  {test_df.shape}")
display(train_df.head(3))

## 3. Visualizing Handwritten Digit Samples

Each row in `train.csv` contains 784 pixel columns ($28 \times 28$). We reshape pixel values into 2D matrices and display sample images.

In [ ]:
labels = train_df['label'].values
pixel_cols = [c for c in train_df.columns if c != 'label']
images = train_df[pixel_cols].values.reshape(-1, 28, 28)

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i], cmap='gray')
    ax.set_title(f"Digit: {labels[i]}", fontsize=10, fontweight='bold')
    ax.axis('off')

plt.suptitle("Sample Handwritten Digit Images (28x28 Grayscale)", fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

## 4. PyTorch Data Pipeline & CNN Architecture

In [ ]:
# Normalize pixel values [0, 255] -> [0.0, 1.0]
X_data = (train_df[pixel_cols].values / 255.0).reshape(-1, 1, 28, 28).astype(np.float32)
y_data = train_df['label'].values.astype(np.int64)

# Train/Validation Split (80/20)
X_train, X_val, y_train, y_val = train_test_split(X_data, y_data, test_size=0.2, random_state=42, stratify=y_data)

# PyTorch DataLoaders
train_dataset = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
val_dataset = TensorDataset(torch.tensor(X_val), torch.tensor(y_val))

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

# PyTorch Convolutional Neural Network (CNN)
class DigitCNN(nn.Module):
    def __init__(self):
        super(DigitCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),  # (32, 28, 28)
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                         # (32, 14, 14)
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1), # (64, 14, 14)
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)                          # (64, 7, 7)
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.25),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )
        
    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

model = DigitCNN().to(device)
print(model)

## 5. Model Training & Epoch Validation Loop

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
epochs = 5

history = {'train_loss': [], 'val_acc': []}

print("🚀 Starting PyTorch CNN Training Loop...")
for epoch in range(1, epochs + 1):
    model.train()
    running_loss = 0.0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, lbls)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        
    epoch_loss = running_loss / len(train_loader.dataset)
    
    # Validation Accuracy
    model.eval()
    correct = 0
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            preds = model(imgs).argmax(dim=1)
            correct += (preds == lbls).sum().item()
            
    val_acc = (correct / len(val_loader.dataset)) * 100.0
    history['train_loss'].append(epoch_loss)
    history['val_acc'].append(val_acc)
    print(f"Epoch [{epoch}/{epochs}] -> Loss: {epoch_loss:.4f} | Validation Accuracy: {val_acc:.2f}%")

## 6. Visual Diagnostics & Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss & Accuracy Plot
axes[0].plot(range(1, epochs + 1), history['train_loss'], marker='o', color='crimson', label='Train Loss')
axes[0].set_title("Training Loss Curve", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("CrossEntropy Loss")
axes[0].legend()

# Confusion Matrix
model.eval()
all_preds = []
all_targets = []
with torch.no_grad():
    for imgs, lbls in val_loader:
        imgs = imgs.to(device)
        preds = model(imgs).argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_targets.extend(lbls.numpy())

cm = confusion_matrix(all_targets, all_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1], cbar=False)
axes[1].set_title("10x10 Digit Classification Confusion Matrix", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Predicted Digit")
axes[1].set_ylabel("True Digit")

plt.tight_layout()
plt.show()

## 7. Kaggle Predictions & Submission Export

In [ ]:
# Preprocess test dataset
test_pixel_cols = [c for c in test_df.columns if c.lower() != 'imageid']
X_test_arr = (test_df[test_pixel_cols].values / 255.0).reshape(-1, 1, 28, 28).astype(np.float32)
test_tensor = torch.tensor(X_test_arr).to(device)

model.eval()
with torch.no_grad():
    test_preds = model(test_tensor).argmax(dim=1).cpu().numpy()

image_ids = test_df['ImageId'].values if 'ImageId' in test_df.columns else np.arange(1, len(test_preds) + 1)

sub_df = pd.DataFrame({
    'ImageId': image_ids,
    'Label': test_preds
})

sub_df.to_csv('submission.csv', index=False)
print(f"💾 Successfully generated 'submission.csv' with {len(sub_df)} predictions!")
display(sub_df.head(10))